# Dashboard for the 13–17 November 2026 – WRF meteo@uniparthenope (d03)
Reads the curated Zarr product created by `wrf_campania_case_study.ipynb`.
Open at `/user/<username>/voila/render/notebooks/wrf_voila_dashboard.ipynb`.

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, clear_output

zarr_path = os.environ.get("WRF_ZARR", "/home/jovyan/work/outputs/wrf_campania/wrf_campania_curated.zarr")
ds = xr.open_zarr(zarr_path)
lat, lon, land = ds.lat.values, ds.lon.values, ds.landmask.values
times = pd.to_datetime(ds.time.values)
CITIES = {"Napoli": (40.852, 14.268), "Salerno": (40.678, 14.765), "Avellino": (40.914, 14.793),
         "Benevento": (41.130, 14.782), "Caserta": (41.072, 14.333), "Positano": (40.628, 14.485),
         "Capri": (40.551, 14.243)}
FIELDS = {"Hourly rain [mm]": ("rain_h", "Blues", 0, 10), "10 m wind [m/s]": ("ws10", "YlOrRd", 0, 25),
         "2 m temperature [°C]": ("t2c", "RdYlBu_r", None, None)}

def nearest(la, lo):
    d = (lat - la) ** 2 + ((lon - lo) * np.cos(np.deg2rad(la))) ** 2
    return np.unravel_index(np.argmin(d), d.shape)

hour = w.SelectionSlider(options=[(f"{t:%d/%m %H}:00", i) for i, t in enumerate(times)],
                        index=len(times) // 2, description="Hour", layout=w.Layout(width="60%"))
field = w.Dropdown(options=list(FIELDS), description="Field")
city = w.Dropdown(options=list(CITIES), value="Positano", description="City")
out = w.Output()

def update(*_):
    var, cmap, vmin, vmax = FIELDS[field.value]
    with out:
        clear_output(wait=True)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.3]})
        m = a1.pcolormesh(lon, lat, ds[var].isel(time=hour.value), cmap=cmap, vmin=vmin, vmax=vmax,
                          shading="auto")
        a1.contour(lon, lat, land.astype(float), levels=[0.5], colors="k", linewidths=.6)
        la, lo = CITIES[city.value]; a1.plot(lo, la, "k*", ms=10)
        fig.colorbar(m, ax=a1, label=field.value); a1.set_aspect("equal")
        a1.set_title(f"{field.value} – {times[hour.value]:%d/%m/%Y %H}:00 UTC")
        j, i = nearest(la, lo)
        s = ds[var].isel(south_north=j, west_east=i)
        if var == "rain_h":
            a2.bar(times, s, width=1 / 24, color="tab:blue")
        else:
            a2.plot(times, s, color="tab:red")
        a2.axvline(times[hour.value], color="k", lw=1)
        a2.set_title(f"{field.value} – {city.value}"); a2.grid(alpha=.3)
        fig.autofmt_xdate(); plt.tight_layout(); plt.show()

for c in (hour, field, city):
    c.observe(update, "value")
display(w.VBox([hour, w.HBox([field, city]), out]))
update()